In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hindi-english-parallel-corpus/hindi_english_parallel.csv


## Tried to implement transformer from scratch on own dataset

In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
import time
from collections import Counter

In [3]:
# data = pd.read_csv('/kaggle/input/hindi-english-parallel-corpus/hindi_english_parallel.csv')
# print(len(data))
# data.head()

In [4]:
# df = data[:1000]
# print(len(df))
# df.head()

In [5]:
# eng = df['english']
# print(type(eng))

# eng = df['english'].astype(str).tolist()
# print(type(eng))

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, V)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        attn_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(attn_output)
        
        return output, attention_weights

In [8]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

In [9]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Self-attention with residual connection and normalization
        attn_output, _ = self.self_attention(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_output))
        
        # Feed forward with residual connection and normalization
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_output))
        
        return x

In [10]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Masked self-attention
        attn_output, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_output))
        
        # Cross-attention with encoder output
        attn_output, _ = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout2(attn_output))
        
        # Feedforward
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_output))
        
        return x

In [11]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=5000, dropout=0.1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.d_model = d_model
        
    def forward(self, x, mask=None):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.positional_encoding(x)
        
        for layer in self.layers:
            x = layer(x, mask)
        
        return x

In [12]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=5000, dropout=0.1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.d_model = d_model
        
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.positional_encoding(x)
        
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        
        return x

In [13]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8,
                 d_ff=2048, num_layers=6, max_len=5000, dropout=0.1):
        super().__init__()
        
        self.encoder = Encoder(src_vocab_size, d_model, num_heads, d_ff, num_layers, max_len, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_heads, d_ff, num_layers, max_len, dropout)
        self.linear = nn.Linear(d_model, tgt_vocab_size)
        
        self._init_parameters()
        
    def _init_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def generate_square_subsequent_mask(self, sz, device):
        """Fixed: Create proper causal mask"""
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return ~mask  # Invert: True where we CAN attend, False where we can't
    
    def create_padding_mask(self, seq, pad_idx=0):
        return (seq != pad_idx).unsqueeze(1).unsqueeze(2)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        encoder_output = self.encoder(src, src_mask)
        decoder_output = self.decoder(tgt, encoder_output, src_mask, tgt_mask)
        output = self.linear(decoder_output)
        return output
    
    def encode(self, src, src_mask=None):
        return self.encoder(src, src_mask)
    
    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)


In [14]:
class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.src_sentences)
    
    def __getitem__(self, idx):
        src = self.src_sentences[idx]
        tgt = self.tgt_sentences[idx]
        
        src_indices = [self.src_vocab['<sos>']] + [self.src_vocab.get(word, self.src_vocab['<unk>']) 
                                                     for word in src.split()] + [self.src_vocab['<eos>']]
        tgt_indices = [self.tgt_vocab['<sos>']] + [self.tgt_vocab.get(word, self.tgt_vocab['<unk>']) 
                                                     for word in tgt.split()] + [self.tgt_vocab['<eos>']]
        
        src_indices = src_indices + [self.src_vocab['<pad>']] * (self.max_len - len(src_indices))
        tgt_indices = tgt_indices + [self.tgt_vocab['<pad>']] * (self.max_len - len(tgt_indices))
        
        src_indices = src_indices[:self.max_len]
        tgt_indices = tgt_indices[:self.max_len]
        
        return torch.tensor(src_indices), torch.tensor(tgt_indices)

In [15]:
def build_vocabulary(sentences, min_freq=1):
    counter = Counter()
    for sentence in sentences:
        counter.update(sentence.split())
    
    vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
    idx = 4
    
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    
    return vocab

In [16]:
def get_sample_data():
    english_sentences = [
        "i am a student", "he is a teacher", "she is a doctor", "we are friends",
        "they are students", "i like programming", "he likes music", "she likes reading",
        "we like sports", "they like movies", "i am learning french", "he is studying english",
        "she is reading a book", "we are playing soccer", "they are watching television",
        "good morning", "good evening", "how are you", "i am fine", "thank you very much",
        "you are welcome", "see you later", "i love you", "i miss you", "have a nice day",
        "this is my house", "that is your car", "these are our books", "those are their pens",
        "i want to learn", "he wants to travel", "she wants to cook", "we want to dance",
        "they want to sing", "the cat is black", "the dog is brown", "the sky is blue",
        "the sun is yellow", "the grass is green", "i can speak english", "he can play guitar",
        "she can write stories", "we can solve problems", "they can build robots",
        "i am very happy","he is very smart","she is very kind","we are ready","they are busy","i like coffee",
        "he likes tea","she likes chocolate","we like travelling","they like shopping","i am eating food",
        "he is drinking water","she is cooking dinner","we are studying together","they are working hard","good night","what is your name",
        "my name is manthan","where are you going","i am going home","please help me","i need your help","do you understand",
        "yes i understand","no i do not understand","i am tired","he is tired","she is tired","we are tired",
        "they are tired","open the door","close the window","sit here","stand there","come here",
        "go there","i am hungry","he is hungry","she is hungry","we are hungry","they are hungry",
    ]
    
    hindi_sentences = [
        "main ek student hoon","woh ek teacher hai","woh ek doctor hai","hum dost hain",
        "woh students hain","mujhe programming pasand hai","use music pasand hai","use padhna pasand hai",
        "hume sports pasand hain","unhe movies pasand hain","main french seekh raha hoon","woh english padh raha hai",
        "woh ek kitab padh rahi hai","hum football khel rahe hain","woh television dekh rahe hain","good morning",
        "good evening","aap kaise ho","main theek hoon","bahut dhanyavaad",
        "aapka swagat hai","phir milenge","main tumse pyaar karta hoon","mujhe tumhari yaad aati hai",
        "aapka din shubh ho","yeh mera ghar hai","woh tumhari gaadi hai","yeh hamari kitabein hain",
        "woh unke pen hain","main seekhna chahta hoon","woh yatra karna chahta hai","woh khana banana chahti hai",
        "hum naachna chahte hain","woh gaana chaahte hain","billi kaali hai","kutta bhura hai",
        "aakash neela hai","suraj peela hai","ghaas hari hai","main english bol sakta hoon",
        "woh guitar baja sakta hai","woh kahaniyan likh sakti hai","hum samasyaayein suljha sakte hain","woh robots bana sakte hain",
        "main bahut khush hoon","woh bahut smart hai","woh bahut dayalu hai","hum taiyaar hain","woh vyast hain",
        "mujhe coffee pasand hai","use chai pasand hai","use chocolate pasand hai","hume ghoomna pasand hai","unhe shopping pasand hai","main khana kha raha hoon",
        "woh paani pee raha hai","woh raat ka khana bana rahi hai","hum saath padh rahe hain","woh mehnat se kaam kar rahe hain","good night","aapka naam kya hai",
        "mera naam manthan hai","aap kahan ja rahe ho","main ghar ja raha hoon","kripya meri madad karo","mujhe tumhari madad chahiye","kya tum samajhte ho",
        "haan main samajhta hoon","nahi main samajhta nahi hoon","main thak gaya hoon","woh thak gaya hai","woh thak gayi hai","hum thak gaye hain",
        "woh thak gaye hain","darwaza kholo","khidki band karo","yahan baitho","wahan khade ho","yahan aao",
        "wahan jao","main bhookha hoon","woh bhookha hai","woh bhookhi hai","hum bhookhe hain","woh bhookhe hain",
    ]

    #english_sentences = df['english'].astype(str).tolist()
    #hindi_sentences = df['hindi'].astype(str).tolist()
    
    train_size = int(0.8 * len(english_sentences))
    return (english_sentences[:train_size], hindi_sentences[:train_size],
            english_sentences[train_size:], hindi_sentences[train_size:])

In [17]:
def train_epoch(model, dataloader, optimizer, criterion, device, clip=1.0):
    model.train()
    total_loss = 0
    
    for src, tgt in dataloader:
        src, tgt = src.to(device), tgt.to(device)
        
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        
        src_mask = model.create_padding_mask(src).to(device)
        # FIXED: Proper causal mask
        tgt_mask = model.generate_square_subsequent_mask(tgt_input.size(1), device)
        tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(0)
        
        optimizer.zero_grad()
        output = model(src, tgt_input, src_mask, tgt_mask)
        
        output = output.reshape(-1, output.size(-1))
        tgt_output = tgt_output.reshape(-1)
        
        loss = criterion(output, tgt_output)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

In [18]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for src, tgt in dataloader:
            src, tgt = src.to(device), tgt.to(device)
            
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            
            src_mask = model.create_padding_mask(src).to(device)
            tgt_mask = model.generate_square_subsequent_mask(tgt_input.size(1), device)
            tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(0)
            
            output = model(src, tgt_input, src_mask, tgt_mask)
            
            output = output.reshape(-1, output.size(-1))
            tgt_output = tgt_output.reshape(-1)
            
            loss = criterion(output, tgt_output)
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

In [19]:
def greedy_decode(model, src, src_mask, max_len, start_token, end_token, device, temperature=1.0):
    model.eval()
    encoder_output = model.encode(src, src_mask)
    tgt = torch.tensor([[start_token]], device=device)
    
    with torch.no_grad():
        for step in range(max_len - 1):
            tgt_mask = model.generate_square_subsequent_mask(tgt.size(1), device)
            output = model.decode(tgt, encoder_output, src_mask, tgt_mask)
            output = model.linear(output)
            
            # Apply temperature
            logits = output[:, -1, :] / temperature
            
            # ANTI-REPETITION: Penalize recently generated tokens
            if step > 0:
                # Get last 3 tokens
                recent_tokens = tgt[0, -min(3, step):].tolist()
                for token in recent_tokens:
                    logits[0, token] -= 2.0  # Repetition penalty
            
            # Get next token
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)  # Sampling instead of argmax
            
            tgt = torch.cat([tgt, next_token], dim=1)
            
            if next_token.item() == end_token:
                break
    
    return tgt


def translate_sentence(model, sentence, src_vocab, tgt_vocab, inv_tgt_vocab, device, max_len=100):
    model.eval()
    tokens = sentence.lower().split()
    src_indices = [src_vocab['<sos>']] + [src_vocab.get(word, src_vocab['<unk>']) 
                                            for word in tokens] + [src_vocab['<eos>']]
    src = torch.tensor(src_indices).unsqueeze(0).to(device)
    src_mask = model.create_padding_mask(src).to(device)
    
    output_indices = greedy_decode(model, src, src_mask, max_len,
                                   start_token=tgt_vocab['<sos>'],
                                   end_token=tgt_vocab['<eos>'],
                                   device=device,
                                   temperature=0.8)  # Lower temperature = more focused
    
    output_tokens = [inv_tgt_vocab[idx.item()] for idx in output_indices[0]]
    output_tokens = [token for token in output_tokens if token not in ['<sos>', '<eos>', '<pad>']]
    
    return ' '.join(output_tokens)

In [20]:
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

#loading data
train_en, train_fr, val_en, val_fr = get_sample_data()
print(f"  Training samples: {len(train_en)}")
print(f"  Validation samples: {len(val_en)}")

# Build vocab
src_vocab = build_vocabulary(train_en)
tgt_vocab = build_vocabulary(train_fr)
inv_tgt_vocab = {idx: word for word, idx in tgt_vocab.items()}

print(f"  Source vocabulary size: {len(src_vocab)}")
print(f"  Target vocabulary size: {len(tgt_vocab)}")

# Create datasets
train_dataset = TranslationDataset(train_en, train_fr, src_vocab, tgt_vocab)
val_dataset = TranslationDataset(val_en, val_fr, src_vocab, tgt_vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

#create model
d_model = 512
num_heads = 8
d_ff = 2048
num_layers = 4
dropout = 0.1

model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    dropout=dropout
).to(device)


criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab['<pad>'])
optimizer = optim.Adam(model.parameters(), lr=0.0002, betas=(0.9, 0.98), eps=1e-9)

#Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)


num_epochs = 100
best_val_loss = float('inf')

print(f"\n{'Epoch':>5} | {'Train Loss':>12} | {'Val Loss':>12} | {'LR':>10} | {'Time':>8}")
print("-" * 65)

for epoch in range(1, num_epochs + 1):
    start_time = time.time()
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step(val_loss)
    
    epoch_time = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr']
    
    if epoch % 20 == 0 or epoch == 1:
        print(f"{epoch:5d} | {train_loss:12.4f} | {val_loss:12.4f} | {current_lr:10.6f} | {epoch_time:6.2f}s")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss

print(f"\nTraining complete! Best validation loss: {best_val_loss:.4f}")


Using device: cuda
  Training samples: 68
  Validation samples: 17
  Source vocabulary size: 125
  Target vocabulary size: 135

Epoch |   Train Loss |     Val Loss |         LR |     Time
-----------------------------------------------------------------
    1 |       6.2940 |       5.0941 |   0.000200 |   1.10s
   20 |       1.1382 |       4.4080 |   0.000200 |   0.24s
   40 |       0.1655 |       4.2337 |   0.000050 |   0.26s
   60 |       0.0402 |       4.3436 |   0.000013 |   0.24s
   80 |       0.0238 |       4.5012 |   0.000006 |   0.24s
  100 |       0.0230 |       4.5403 |   0.000002 |   0.24s

Training complete! Best validation loss: 3.6941


In [21]:
# Saving model and vocab
torch.save({
    "model_state": model.state_dict(),
    "src_vocab": src_vocab,
    "tgt_vocab": tgt_vocab
}, "translator.pth")

print("\nModel saved as translator.pth")


Model saved as translator.pth


In [22]:
test_sentences = [
    "we like travelling","they like shopping","drink the water","turn on the fan","open the door",
    "close the window",
]
    
for eng_sentence in test_sentences:
    translation = translate_sentence(model, eng_sentence, src_vocab, tgt_vocab, inv_tgt_vocab, device)
    
    try:
        idx = train_en.index(eng_sentence) if eng_sentence in train_en else val_en.index(eng_sentence)
        expected = train_fr[idx] if eng_sentence in train_en else val_fr[idx]
    except:
        expected = "N/A"
    
    print(f"\nEnglish:    {eng_sentence}")
    print(f"Predicted:  {translation}")
    print(f"Expected:   {expected}")


English:    we like travelling
Predicted:  hume ghoomna pasand hai
Expected:   hume ghoomna pasand hai

English:    they like shopping
Predicted:  unhe shopping pasand hai
Expected:   unhe shopping pasand hai

English:    drink the water
Predicted:  woh raat ka khana bana rahi hai
Expected:   N/A

English:    turn on the fan
Predicted:  mujhe tumhari sakta hai
Expected:   N/A

English:    open the door
Predicted:  woh english sakta hai
Expected:   darwaza kholo

English:    close the window
Predicted:  woh raat ka programming hai
Expected:   khidki band karo


In [23]:
while True:
    sentence = input("\nEnter English sentence: ").lower()
    
    if sentence == "exit":
        break
    
    translation = translate_sentence(model, sentence, src_vocab, tgt_vocab, inv_tgt_vocab, device)
    print("Hindi:", translation)


Enter English sentence:  she is cooking


Hindi: woh raat ka khana bana rahi hai



Enter English sentence:  the dog is barking


Hindi: kutta bhura hai



Enter English sentence:  he wants to sing


...


Enter English sentence:  the sky is blue


Hindi: aakash neela hai



Enter English sentence:  the moon is white


Hindi: woh tumhari gaadi hai



Enter English sentence:  exit


# use saved model for future testing

In [24]:
import torch
from model import Transformer              # your Transformer class
from utils import translate_sentence       # your translate function

# Load checkpoint
checkpoint = torch.load("translator.pth", map_location="cpu")

src_vocab = checkpoint["src_vocab"]
tgt_vocab = checkpoint["tgt_vocab"]
inv_tgt_vocab = {idx: word for word, idx in tgt_vocab.items()}

# Recreate model (same hyperparameters)
model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=4,
    dropout=0.1
)

model.load_state_dict(checkpoint["model_state"])
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("=" * 60)
print("Interactive English → Hindi Translator")
print("Type 'exit' to quit")
print("=" * 60)

while True:
    sentence = input("\nEnter English sentence: ").lower()
    
    if sentence == "exit":
        break
    
    translation = translate_sentence(
        model, sentence, src_vocab, tgt_vocab, inv_tgt_vocab, device
    )
    
    print("Hindi:", translation)

ModuleNotFoundError: No module named 'model'